# 🌟 MLSQL — Fine-Tune Sarvam-2b for Malayalam → SQL

This notebook fine-tunes the **Sarvam-2b** model on the **Spider dataset** to convert **Malayalam natural language** to **SQL queries** using **QLoRA** (4-bit quantized LoRA).

## Pipeline
1. ✅ Install dependencies
2. ✅ Download Spider dataset (7,000 training examples)
3. ✅ Translate English questions → Malayalam
4. ✅ Format data for instruction tuning
5. 🚀 Fine-tune Sarvam-2b with QLoRA
6. 🧠 Test inference
7. 💾 Download fine-tuned model

---

> **⚠️ Make sure GPU is enabled:** Go to `Runtime > Change runtime type > T4 GPU`

## Step 0: Check GPU & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Install required packages
!pip install -q datasets transformers peft trl bitsandbytes accelerate
!pip install -q deep-translator sentencepiece protobuf tqdm

## Step 1: Download Spider Dataset

In [ ]:
import json
import os
from datasets import load_dataset

# Create data directory
os.makedirs("data", exist_ok=True)

# Download Spider dataset from HuggingFace
print("Downloading Spider dataset...")
dataset = load_dataset("xlangai/spider", trust_remote_code=True)

print(f"Train split: {len(dataset['train'])} examples")
print(f"Validation split: {len(dataset['validation'])} examples")

# Collect unique databases
unique_dbs = set(row["db_id"] for row in dataset["train"])
print(f"Unique databases: {len(unique_dbs)}")

# Save training records
records = []
for row in dataset["train"]:
    records.append({
        "question": row["question"],
        "query": row["query"],
        "db_id": row["db_id"],
        "schema_context": ""
    })

with open("data/spider_train.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved {len(records)} records to data/spider_train.json")

# Show samples
for i in range(3):
    print(f"\n[{i+1}] Q: {records[i]['question']}")
    print(f"    SQL: {records[i]['query']}")
    print(f"    DB: {records[i]['db_id']}")

## Step 2: Translate English → Malayalam

Uses `deep-translator` (Google Translate) as a free translation backend.

> 💡 If you have a **Sarvam AI API key**, set it below for higher quality translations.

In [ ]:
# OPTIONAL: Set your Sarvam AI API key for better translations
SARVAM_API_KEY = ""  # Paste your key here, or leave empty to use free Google Translate

In [ ]:
import time
import requests
from tqdm.notebook import tqdm
from deep_translator import GoogleTranslator

# Load data
with open("data/spider_train.json", "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")

# --- Translation functions ---

def translate_sarvam(text, api_key):
    """Translate using Sarvam AI API."""
    headers = {"Content-Type": "application/json", "API-Subscription-Key": api_key}
    payload = {
        "input": text,
        "source_language_code": "en-IN",
        "target_language_code": "ml-IN",
        "model": "mayura:v1",
        "enable_preprocessing": True
    }
    for attempt in range(3):
        try:
            resp = requests.post("https://api.sarvam.ai/translate", json=payload, headers=headers, timeout=30)
            if resp.status_code == 200:
                return resp.json().get("translated_text", text)
            elif resp.status_code == 429:
                time.sleep((attempt + 1) * 2)
            else:
                return None
        except Exception:
            time.sleep(1)
    return None

def translate_google(text):
    """Translate using Google Translate (free)."""
    try:
        return GoogleTranslator(source='en', target='ml').translate(text)
    except Exception as e:
        print(f"  Error: {e}")
        return None

# --- Run translation ---
use_sarvam = bool(SARVAM_API_KEY)
print(f"Using: {'Sarvam AI API' if use_sarvam else 'Google Translate (free)'}")

# Checkpoint support
checkpoint_file = "data/translation_checkpoint.json"
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    print(f"Resuming from checkpoint: {checkpoint['completed']} done")
else:
    checkpoint = {"completed": 0, "translations": {}}

start_idx = checkpoint["completed"]

for i in tqdm(range(start_idx, len(records)), initial=start_idx, total=len(records), desc="Translating"):
    question = records[i]["question"]

    if str(i) in checkpoint["translations"]:
        continue

    if use_sarvam:
        translated = translate_sarvam(question, SARVAM_API_KEY)
        time.sleep(0.2)
    else:
        translated = translate_google(question)
        time.sleep(0.5)

    checkpoint["translations"][str(i)] = translated if translated else question
    checkpoint["completed"] = i + 1

    # Save checkpoint every 100 items
    if (i + 1) % 100 == 0:
        with open(checkpoint_file, "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False)
        # Also save partial output
        print(f"  Checkpoint: {i+1}/{len(records)}")

# Save final checkpoint
with open(checkpoint_file, "w", encoding="utf-8") as f:
    json.dump(checkpoint, f, ensure_ascii=False)

# Build output
output_records = []
for i, record in enumerate(records):
    output_records.append({
        "question_en": record["question"],
        "question_ml": checkpoint["translations"].get(str(i), record["question"]),
        "query": record["query"],
        "db_id": record["db_id"],
        "schema_context": record.get("schema_context", "")
    })

with open("data/spider_malayalam.json", "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved {len(output_records)} translated records to data/spider_malayalam.json")

# Show samples
for i in range(5):
    print(f"\n[{i+1}] EN: {output_records[i]['question_en']}")
    print(f"     ML: {output_records[i]['question_ml']}")
    print(f"     SQL: {output_records[i]['query']}")

# Cleanup checkpoint
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)

## Step 3: Format Training Data

Converts translated data into instruction-tuning format for SFTTrainer.

In [ ]:
import random

# Load translated data
with open("data/spider_malayalam.json", "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")

# Instruction template
TEMPLATE = """### Instruction:
Convert the following Malayalam question to an SQL query.

### Question:
{question}

### SQL:
{sql}"""

TEMPLATE_WITH_SCHEMA = """### Instruction:
Convert the following Malayalam question to an SQL query based on the given database schema.

### Schema:
{schema}

### Question:
{question}

### SQL:
{sql}"""

# Format records
formatted = []
for rec in records:
    if not rec.get("question_ml") or not rec.get("query"):
        continue
    schema = rec.get("schema_context", "").strip()
    if schema:
        text = TEMPLATE_WITH_SCHEMA.format(schema=schema, question=rec["question_ml"], sql=rec["query"])
    else:
        text = TEMPLATE.format(question=rec["question_ml"], sql=rec["query"])
    formatted.append({"text": text})

# Shuffle and split
random.seed(42)
random.shuffle(formatted)
val_size = int(len(formatted) * 0.05)
train_data = formatted[val_size:]
val_data = formatted[:val_size]

# Save as JSONL
with open("data/train_dataset.jsonl", "w", encoding="utf-8") as f:
    for rec in train_data:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open("data/val_dataset.jsonl", "w", encoding="utf-8") as f:
    for rec in val_data:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✅ Train: {len(train_data)} | Val: {len(val_data)}")

# Show a sample
print(f"\n{'='*60}")
print("Sample training prompt:")
print(f"{'='*60}")
print(train_data[0]["text"])

## Step 4: Fine-Tune Sarvam-2b with QLoRA 🚀

This is the core step. Uses:
- **4-bit quantization** (NF4) to fit the 2B model in T4 GPU memory
- **LoRA** adapters targeting all projection layers
- **SFTTrainer** from TRL for supervised fine-tuning

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ============================================
# CONFIGURATION - Adjust as needed
# ============================================
BASE_MODEL = "sarvamai/sarvam-2b-v0.5"
OUTPUT_DIR = "sarvam-malayalam-sql-lora"

# Training hyperparameters
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 512
GRADIENT_ACCUMULATION = 4

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

print(f"Model: {BASE_MODEL}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load dataset
print("Loading training data...")
dataset = load_dataset("json", data_files={
    "train": "data/train_dataset.jsonl",
    "validation": "data/val_dataset.jsonl"
})
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Load model with 4-bit quantization (QLoRA)
print("Loading model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("✅ Model loaded!")

In [ ]:
# Apply LoRA adapters
print("Applying LoRA adapters...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print parameter stats
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Configure training
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

print("✅ Trainer ready!")
print(f"Total training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'calculating...'}")

In [ ]:
# 🚀 START TRAINING
print("="*60)
print("TRAINING STARTED")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)

In [ ]:
# Save the fine-tuned LoRA adapter
FINAL_PATH = f"{OUTPUT_DIR}/final"
trainer.model.save_pretrained(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
print(f"✅ LoRA adapter saved to: {FINAL_PATH}")

## Step 5: Test Inference 🧠

Test the fine-tuned model with sample Malayalam questions.

In [ ]:
from peft import PeftModel

# Load base model + LoRA adapter for inference
print("Loading model for inference...")

# Reload base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Load LoRA adapter
inference_model = PeftModel.from_pretrained(base_model, FINAL_PATH)
inference_model.eval()
print("✅ Model ready for inference!")

In [ ]:
def generate_sql(question, schema="", max_new_tokens=256):
    """Generate SQL from a Malayalam question."""
    if schema.strip():
        prompt = f"""### Instruction:
Convert the following Malayalam question to an SQL query based on the given database schema.

### Schema:
{schema}

### Question:
{question}

### SQL:
"""
    else:
        prompt = f"""### Instruction:
Convert the following Malayalam question to an SQL query.

### Question:
{question}

### SQL:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(inference_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    sql = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Clean up
    for stop in ["\n###", "\n\n", "### "]:
        if stop in sql:
            sql = sql[:sql.index(stop)]

    return sql.strip()

print("generate_sql() function ready!")

In [ ]:
# Test with sample Malayalam questions
test_questions = [
    "വിദ്യാര്ത്ഥികളുടെ പേരുകള്‍ കാണിക്കുക",           # Show names of students
    "56-ന് മുകളില്‍ പ്രായമുള്ള വിഭാഗ തലവന്മാര്‍ എത്ര?",  # How many department heads older than 56?
    "പേര്‍ അനുസരിച്ച് വിഭാഗങ്ങള്‍ ക്രമീകരിക്കുക",      # Sort departments by name
    "ആകെ വിദ്യാര്ത്ഥികള്‍ എത്ര?",                        # How many total students?
    "സിനിമകളുടെ പേരുകളും വിലകളും കാണിക്കുക",   # Show movie names and prices
]

print("=" * 60)
print("INFERENCE TEST RESULTS")
print("=" * 60)

for q in test_questions:
    sql = generate_sql(q)
    print(f"\n🗣️  {q}")
    print(f"📝  {sql}")
    print("-" * 60)

## Step 6: Download Fine-Tuned Model 💾

Download the LoRA adapter to use in your local project.

In [ ]:
# Zip the LoRA adapter for download
import shutil

zip_path = shutil.make_archive("sarvam-malayalam-sql-lora", "zip", FINAL_PATH)
print(f"✅ Model zipped: {zip_path}")
print(f"\n📌 To use locally:")
print(f"  1. Download the zip file from the Files panel (📁 icon on the left)")
print(f"  2. Extract to: finetuning/sarvam-malayalam-sql-lora/final/")
print(f"  3. Run: python 05_inference.py --serve")

In [ ]:
# Alternative: Push to HuggingFace Hub (optional)
# Uncomment and set your token to push the adapter to HuggingFace

# from huggingface_hub import login
# login(token="your_hf_token_here")
# trainer.model.push_to_hub("your-username/sarvam-malayalam-sql-lora")
# tokenizer.push_to_hub("your-username/sarvam-malayalam-sql-lora")
# print("✅ Pushed to HuggingFace Hub!")

---

## ✅ Done!

### Next Steps
1. **Download** the `sarvam-malayalam-sql-lora.zip` file
2. **Extract** it to your project: `finetuning/sarvam-malayalam-sql-lora/final/`
3. **Start inference server**: `python 05_inference.py --serve`
4. **Start Spring Boot**: `mvn spring-boot:run`
5. **Test**: Send a Malayalam question to `POST /api/malayalam-to-nosql`

### Full Pipeline
```
Malayalam Question → [Sarvam-2b + LoRA] → SQL → [JSQLParser] → MongoDB Query
```